# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates loading and exploration of the FAIR² dataset, using the `mlcroissant` library with full traceability via Croissant schema `@id`s for all entities.

### Dataset Source
The dataset schema is available at:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This will fetch the Croissant schema, parse available record sets, and allow for further processing.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Croissant schema URL (the FAIR² dataset)
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is an object

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review all available record sets, fields, and their `@id`'s and names. All references to dataset entities are via their `@id`.

In [ ]:
# Explore record sets available in the metadata schema
record_set_objs = metadata.record_sets

print("Available Record Sets:")
record_set_ids = []
for rs in record_set_objs:
    print(f"- RecordSet @id: {rs.id}, name: {rs.name}")
    record_set_ids.append(rs.id)
    # List fields for this record set
    if hasattr(rs, 'fields'):
        for f in rs.fields:
            print(f"    Field @id: {f.id}, name: {f.name}, dataType: {getattr(f, 'data_type', None)}")
    if hasattr(rs, 'columns'):
        for col in rs.columns:
            print(f"    Column @id: {col.id}, name: {col.name}, dataType: {getattr(col, 'data_type', None)}")

## 3. Data Extraction
Load tabular data from each record set into `pandas` DataFrames. Reference record sets and fields by their `@id` as above.

In [ ]:
# Utility: Display short summary of all records in each record set
dataframes = {}

for rs_id in record_set_ids:
    print(f"\nLoading data from RecordSet @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(2).to_string(index=False))

## 4. Exploratory Data Analysis (EDA)
Process data from a relevant record set of interest. Apply filtering, normalization, and grouping. Choose numeric and categorical fields by their `@id` (referenced from the overview above).

In [ ]:
# For demonstration, we select the first record set and a numeric/categorical field therein
# Update these @ids as appropriate for your analysis based on the overview section above.

# Example: pick first RecordSet and its first two fields
example_rs_id = record_set_ids[0]
df = dataframes[example_rs_id]

# Print columns and pick a numeric and group field (by @id)
print(f"Fields in RecordSet {example_rs_id}: {df.columns.tolist()}")

# Let's find a numeric field and a group field candidate by scanning column names
# We'll filter for numeric-like columns and group-by candidates heuristically
numeric_field = None
group_field = None

for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break

for col in df.columns:
    if col != numeric_field and (df[col].dtype == object or df[col].dtype == 'category'):
        group_field = col
        break

print(f"Selected numeric field: {numeric_field}")
print(f"Selected group (categorical) field: {group_field}")

if numeric_field:
    # Remove extreme outliers (values outside 3 stddev)
    mean = df[numeric_field].mean()
    std = df[numeric_field].std()
    filtered_df = df[(df[numeric_field] <= mean + 3*std) & (df[numeric_field] >= mean - 3*std)]

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()

    print(f"\nFiltered and normalized '{numeric_field}' (with outliers removed):")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head(5).to_string(index=False))

    # Group by the chosen group field (if available)
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped mean of '{numeric_field}' by '{group_field}':")
        print(grouped_df.head(10))
else:
    print("No numeric field found in this record set for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship to the grouping field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field], kde=True, color='b')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

    if group_field:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("Visualization skipped: no numeric field available.")

## 6. Conclusion
In this notebook, we loaded clinicopathological data on second primary colorectal cancer survivors via the Croissant schema and `mlcroissant`. We inspected record sets and fields using only `@id` references, loaded tables into DataFrames, applied simple filtering/normalization, and visualized numeric distributions by group.

For further analysis, consult field and variable definitions in the Croissant schema and apply domain-specific transformations as needed. The `mlcroissant` library and Croissant `@id` referencing enable transparent, reproducible, and FAIR-compliant data exploration.